# **Linkedin Dataset Ingestion and Mapping**

## **1. Introduction**
The Linkedin dataset requires additional mapping to bring in necessary features provided by the authors.

### **Linkedin Job Postings Dataset**
[LinkedIn Job Postings (2023 - 2024)](https://www.kaggle.com/datasets/arshkon/linkedin-job-postings) by [Arsh Koneru](https://www.kaggle.com/arshkon) and [Zoey Yu Zou](https://www.kaggle.com/zoeyyuzou)

In [1]:
import pandas as pd
from pandarallel import pandarallel
from jobrec import config

## **2. Data Mapping**

In [2]:
# Load raw unmapped dataset
jobs_df = pd.read_csv(config.RAW_DATA_DIR/'jobs_raw.csv')

In [3]:
# Load in auxilliary datasets
skill_ids    = pd.read_csv(config.RAW_DATA_DIR/'jobs'/'job_skills.csv')
industry_ids = pd.read_csv(config.RAW_DATA_DIR/'jobs'/'job_industries.csv')

In [4]:
# Load in mappings
mapped_skills      = pd.read_csv(config.RAW_DATA_DIR/'mappings'/'skills.csv')
mapped_industries  = pd.read_csv(config.RAW_DATA_DIR/'mappings'/'industries.csv')

In [5]:
# Merge mapped skills with associated ids
skill_ids    = skill_ids.merge(mapped_skills, on='skill_abr', how='left')
industry_ids = industry_ids.merge(mapped_industries, on='industry_id', how='left')

In [6]:
# Collapse to lists of unique mapped skill names per job_id
skills_per_job = skill_ids.groupby('job_id')['skill_name'].apply(lambda x: list(set(x.dropna().str.lower()))).reset_index()

In [7]:
# Collapse to lists of unique mapped industry names per job_id
industries_per_job = industry_ids.groupby('job_id')['industry_name'].apply(lambda x: list(set(x.dropna().str.lower()))).reset_index()

In [8]:
# Merge in mapped skills and industries
jobs_df = jobs_df.merge(skills_per_job, on='job_id', how='left')
jobs_df = jobs_df.merge(industries_per_job, on='job_id', how='left')

In [9]:
# Save dataframe to pickle
jobs_df.to_pickle(config.RAW_DATA_DIR/'jobs_raw.pkl')